[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_7_deep_learning_6d_pose/21_foundationpose_freeze.ipynb)

# Notebook 21 — FoundationPose & FreeZe: Zero-Shot 6D Pose Estimation

---

**Part 7 · Deep Learning 6D Pose**

```
┌─────────────────────────────────────────────────────────────────────┐
│   6D Pose WITHOUT Training                                          │
│                                                                     │
│   RGB-D Image ──► Foundation Model ──► Feature Fusion ──► Pose     │
│   CAD Model   ──►  Geometric Feats ──► RANSAC Align  ──► [R|t]     │
└─────────────────────────────────────────────────────────────────────┘
```

> **The big idea**: The newest pose estimation methods use pretrained **foundation models** and **geometric registration** to estimate 6D pose for objects they have *never seen during training*. No custom dataset. No retraining. Just a CAD model.

## Recommended Watch

> Watch **both** before opening this notebook — they cover the two methods (FoundationPose and FreeZe) implemented here.

| # | Title | Link | Duration | Note |
|---|---|---|---|---|
| 1 | **6D Pose Estimation WITHOUT MARKERS for 3D Object Detection via FoundationPose & EfficientPose** | [▶ Watch](https://www.youtube.com/watch?v=mlXs5kIQ5p4) | ~20 min | Focus on the FoundationPose section (second half) |
| 2 | **3D Object Detection (6D Pose Estimation) without Training using FreeZe** | [▶ Watch](https://www.youtube.com/watch?v=Mgmt93kXK_4) | ~15 min | Dedicated FreeZe video — two-branch architecture explained |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-python open3d -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import cv2
print('Core imports OK')

## Learning Objectives

By the end of this notebook you will:

1. Understand the **full 6D pose estimation spectrum** from markers → EfficientPose → FoundationPose → FreeZe  
2. Explain FoundationPose's **4 capabilities** (model-based/model-free × estimation/tracking)  
3. Trace through the **FreeZe two-branch architecture** (visual + geometric features)  
4. Understand why **RANSAC** is used for the final alignment step  
5. Know what inputs each method requires and when to use which  
6. Simulate a zero-shot pose result using Python

## 1 — The Complete 6D Pose Estimation Spectrum

We've built up the full toolbox across this course:

```
 CLASSICAL ◄──────────────────────────────────────────► MODERN

 ArUco + PnP    Feature Matching    EfficientPose    FoundationPose    FreeZe
     │                │                  │                │               │
  Markers         Known pts          Trained DL       Foundation      Zero-shot
  required        + CAD              on object        model           geometric
     │                │                  │                │               │
  Very fast        Medium             Fast GPU        Fast GPU        No GPU needed
  Deterministic    Fragile            New obj=retrain  Generalizes     Unseen objects
```

**This notebook covers the right end of the spectrum**: methods that work on objects they have *never seen during training*.

In [ ]:
# Visualize the tradeoff space
methods = ['ArUco\n+PnP', 'Feature\nMatching', 'EfficientPose', 'Foundation\nPose', 'FreeZe']
setup_effort = [1, 4, 8, 6, 5]        # setup cost (1=easy, 10=hard)
generalization = [1, 3, 4, 8, 9]      # works on new objects (1=poor, 10=great)
real_time = [10, 6, 8, 7, 6]          # real-time speed (1=slow, 10=fast)
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: setup effort vs generalization
ax = axes[0]
for i, (m, se, gen, c) in enumerate(zip(methods, setup_effort, generalization, colors)):
    ax.scatter(se, gen, s=300, color=c, zorder=5)
    ax.annotate(m, (se, gen), textcoords='offset points', xytext=(8, 0),
                fontsize=8, va='center')
ax.set_xlabel('Setup Effort (1=easy, 10=hard)', fontsize=11)
ax.set_ylabel('Generalization to New Objects', fontsize=11)
ax.set_title('Pose Methods: Setup vs Generalization', fontweight='bold')
ax.set_xlim(0, 11); ax.set_ylim(0, 11)
ax.grid(True, alpha=0.3)

# Bar: real-time score
ax2 = axes[1]
bars = ax2.bar(methods, real_time, color=colors, edgecolor='black', linewidth=0.7)
ax2.set_ylabel('Real-Time Score', fontsize=11)
ax2.set_title('Speed / Real-Time Capability', fontweight='bold')
ax2.set_ylim(0, 12)
for bar, val in zip(bars, real_time):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.2, str(val),
             ha='center', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.suptitle('6D Pose Method Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 2 — FoundationPose

FoundationPose (NVIDIA, 2024) is a **unified framework** that handles both estimation and tracking for both model-based and model-free scenarios.

### The 4 Capabilities

| Mode | Input | Description |
|------|-------|-------------|
| **Model-based Estimation** | CAD model + RGB-D | First-frame 6D pose from known 3D shape |
| **Model-based Tracking** | CAD model + video | Continuous pose tracking using shape |
| **Model-free Estimation** | Reference images + RGB-D | First-frame pose from a few photos |
| **Model-free Tracking** | Reference images + video | Continuous tracking from photos only |

Most older models only do **one** of these. FoundationPose does all four.

### Why This Matters for Robotics

```
Warehouse robot picks up a new product:

  OLD way:  1) CAD model → 2) collect dataset → 3) retrain → 4) deploy  (weeks)
  NEW way:  1) CAD model or 3 photos → 2) run FoundationPose           (minutes)
```

## 3 — FoundationPose Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     FoundationPose Pipeline                                 │
│                                                                             │
│  Inputs:                                                                    │
│  ┌─────────────┐  ┌──────────────┐  ┌───────────────────────────────────┐  │
│  │ RGB Image   │  │ Depth Image  │  │ CAD Model OR Reference Images     │  │
│  └──────┬──────┘  └──────┬───────┘  └──────────────┬────────────────────┘  │
│         │                │                          │                       │
│         └────────────────┘                          │                       │
│                  │                                  │                       │
│                  ▼                                  ▼                       │
│         ┌────────────────┐               ┌───────────────────┐             │
│         │ Neural Object  │               │ Render Hypotheses │             │
│         │ Modeling (NeRF-│               │ (many viewpoints) │             │
│         │ like views)    │               └─────────┬─────────┘             │
│         └───────┬────────┘                         │                       │
│                 │                                  │                       │
│                 └──────────────────────────────────┘                       │
│                                    │                                        │
│                                    ▼                                        │
│                        ┌───────────────────────┐                           │
│                        │ Transformer Encoder   │                           │
│                        │ (Self-Attention)       │                           │
│                        │ Rank Pose Hypotheses  │                           │
│                        └───────────┬───────────┘                           │
│                                    │                                        │
│                                    ▼                                        │
│                           ┌────────────────┐                               │
│                           │ Best 6D Pose   │                               │
│                           │   [R | t]      │                               │
│                           └────────────────┘                               │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Key Components

**Neural Object Modeling** — similar to NeRF: given a few views, reconstruct a neural 3D representation of the object. This means you don't need a perfect CAD model; photos suffice.

**Pose Hypothesis Generation** — instead of predicting one pose, generate *many candidate poses* from different viewpoints.

**Transformer-based Ranking** — a self-attention transformer looks at all candidates and selects the best one. This is robust to ambiguous or occluded views.

### Synthetic Training Data

FoundationPose avoids the expensive labeling problem by generating training data synthetically:

```
Text prompt ──► Diffusion model ──► 3D object ──► Physics engine ──► Rendered training frames
```

This allows training on millions of diverse objects without any manual annotation.

In [ ]:
# Simulate the hypothesis generation + ranking concept

np.random.seed(42)

def euler_to_R(rx, ry, rz):
    """Simple Euler → rotation matrix (ZYX convention)."""
    cx, sx = np.cos(rx), np.sin(rx)
    cy, sy = np.cos(ry), np.sin(ry)
    cz, sz = np.cos(rz), np.sin(rz)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return Rz @ Ry @ Rx

# Ground truth pose
R_gt = euler_to_R(0.3, 0.5, 0.2)
t_gt = np.array([0.15, -0.08, 0.42])

# Generate N pose hypotheses with noise
N_hypotheses = 50
hypotheses = []
scores = []

for i in range(N_hypotheses):
    noise_r = np.random.randn(3) * 0.5   # rotation noise (rad)
    noise_t = np.random.randn(3) * 0.08  # translation noise (m)
    R_hyp = euler_to_R(0.3 + noise_r[0], 0.5 + noise_r[1], 0.2 + noise_r[2])
    t_hyp = t_gt + noise_t
    hypotheses.append((R_hyp, t_hyp))
    # Score = inverse of combined rotation + translation error (mock transformer output)
    rot_err = np.linalg.norm(noise_r)
    t_err = np.linalg.norm(noise_t)
    score = np.exp(-(rot_err + 10 * t_err))  # pretend transformer scoring
    scores.append(score)

scores = np.array(scores)
best_idx = np.argmax(scores)
R_best, t_best = hypotheses[best_idx]

print(f'Generated {N_hypotheses} pose hypotheses')
print(f'Best hypothesis index: {best_idx}  (score = {scores[best_idx]:.4f})')
print(f'\nGround-truth translation: {t_gt}')
print(f'Best hypothesis translation: {t_best.round(4)}')
print(f'Translation error: {np.linalg.norm(t_best - t_gt)*100:.2f} cm')

# Plot score distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N_hypotheses), scores, color=['#e74c3c' if i == best_idx else '#3498db'
                                             for i in range(N_hypotheses)], edgecolor='black', lw=0.5)
ax.set_xlabel('Hypothesis Index', fontsize=11)
ax.set_ylabel('Transformer Score', fontsize=11)
ax.set_title('FoundationPose: Hypothesis Ranking (red = selected best)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4 — FreeZe: Training-Free Zero-Shot 6D Pose

**FreeZe** = *Training-Free Zero-Shot 6D Pose Estimation with Geometric and Vision Foundation Models*

### The Problem it Solves

| Approach | Needs Training | Works on Unseen Objects |
|----------|---------------|------------------------|
| EfficientPose | Yes (per object class) | No |
| FoundationPose | Yes (large-scale, once) | Limited |
| **FreeZe** | **No** | **Yes** |

FreeZe uses two inputs you *already have*:
1. A **3D CAD model** of your object (`.obj` or point cloud)
2. An **RGB-D image** of the scene

And it uses features from **pretrained foundation models** — it never trains any pose-specific weights.

## 5 — FreeZe Architecture: Two-Branch Feature Fusion

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                         FreeZe Pipeline                                      │
│                                                                              │
│   LEFT BRANCH (Model Side)          RIGHT BRANCH (Scene Side)               │
│                                                                              │
│   3D CAD Model                      RGB-D Image                             │
│        │                                 │                                   │
│        ▼                                 ▼                                   │
│   Render 2D views              Crop ROI around object                        │
│        │                                 │                                   │
│        ▼                                 ▼                                   │
│   Vision Foundation         Vision Foundation Model                          │
│   Model (DINO-like)         (DINO-like)                                     │
│        │                                 │                                   │
│        ▼                                 ▼                                   │
│   2D features               2D features                                      │
│        │                                 │                                   │
│        ▼                                 ▼                                   │
│   Lift to 3D               Lift to 3D (using depth)                          │
│   (from CAD geometry)                   │                                   │
│        │                                 │                                   │
│   Geometric features        Geometric features                               │
│   (textureless cloud)       (from depth cloud)                              │
│        │                                 │                                   │
│        ▼                                 ▼                                   │
│   Fuse visual +             Fuse visual +                                    │
│   geometric features        geometric features                               │
│        │                                 │                                   │
│        └─────────────┬───────────────────┘                                  │
│                      │                                                       │
│                      ▼                                                       │
│              Feature Matching                                                │
│               (find correspondences)                                         │
│                      │                                                       │
│                      ▼                                                       │
│               RANSAC Registration                                            │
│                      │                                                       │
│                      ▼                                                       │
│                 6D Pose [R | t]                                              │
└──────────────────────────────────────────────────────────────────────────────┘
```

### Why Two Types of Features?

| Feature Type | What it Captures | Strength |
|-------------|-----------------|----------|
| **Visual features** (DINO) | Texture, color, appearance | Works in textured regions |
| **Geometric features** | Edges, normals, curvature | Works in textureless regions |
| **Fused** | Both | Robust to both cases |

By combining both, FreeZe handles objects that are textureless (e.g., metal parts) AND textured (e.g., branded boxes).

## 6 — RANSAC: Why It's Used for Final Alignment

After feature matching, we have a set of 3D–3D correspondences:

```
  Point on CAD model  ──corresponds to──  Point in scene point cloud
  p1_model ─────────────────────────────► p1_scene
  p2_model ─────────────────────────────► p2_scene
  ...                                     ...
```

**But feature matching is noisy** — some matches are wrong (outliers).

**RANSAC** (Random Sample Consensus) handles this:

```
Repeat many times:
  1. Randomly sample 3 correspondences
  2. Compute rigid transform [R | t] from these 3 points
  3. Count how many OTHER correspondences agree (inliers)
  4. Keep the [R | t] with the most inliers

Final answer: [R | t] that explains the most correspondences
```

RANSAC is **deterministic in the limit** and **robust to 50%+ outliers**. That's why it's the perfect choice here.

In [ ]:
# Simulate RANSAC-based 3D-3D alignment

def compute_rigid_transform(src, dst):
    """
    Compute [R | t] that maps src → dst using SVD (Kabsch algorithm).
    src, dst: (N, 3) arrays of 3D point correspondences.
    """
    src_c = src - src.mean(axis=0)
    dst_c = dst - dst.mean(axis=0)
    H = src_c.T @ dst_c
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:  # fix reflection
        Vt[-1] *= -1
        R = Vt.T @ U.T
    t = dst.mean(axis=0) - R @ src.mean(axis=0)
    return R, t

def ransac_pose(src_pts, dst_pts, n_iterations=500, inlier_threshold=0.01):
    """
    RANSAC 3D-3D pose estimation.
    Returns best (R, t) and inlier mask.
    """
    best_R, best_t = None, None
    best_inliers = 0
    best_mask = None
    n = len(src_pts)

    for _ in range(n_iterations):
        idx = np.random.choice(n, 3, replace=False)
        try:
            R, t = compute_rigid_transform(src_pts[idx], dst_pts[idx])
        except Exception:
            continue
        transformed = (R @ src_pts.T).T + t
        errors = np.linalg.norm(transformed - dst_pts, axis=1)
        mask = errors < inlier_threshold
        n_inliers = mask.sum()
        if n_inliers > best_inliers:
            best_inliers = n_inliers
            best_R, best_t = R, t
            best_mask = mask

    return best_R, best_t, best_mask

# Create a fake point cloud of a box
np.random.seed(7)
model_pts = np.random.rand(80, 3) * 0.2 - 0.1   # CAD model points

# True pose to recover
R_true = euler_to_R(0.4, 0.2, 0.6)
t_true = np.array([0.1, 0.05, 0.3])

# Scene points = model transformed + noise + 30% outliers
scene_pts_clean = (R_true @ model_pts.T).T + t_true
scene_pts = scene_pts_clean + np.random.randn(*scene_pts_clean.shape) * 0.003

# Add 30% outlier correspondences
n_outliers = 24
outlier_idx = np.random.choice(len(model_pts), n_outliers, replace=False)
scene_pts[outlier_idx] = np.random.rand(n_outliers, 3) * 0.4  # random outliers

# Run RANSAC
R_est, t_est, inlier_mask = ransac_pose(model_pts, scene_pts, n_iterations=1000, inlier_threshold=0.015)

# Compute errors
t_err = np.linalg.norm(t_est - t_true) * 100  # cm
rot_diff = R_true @ R_est.T
angle_err = np.degrees(np.arccos(np.clip((np.trace(rot_diff) - 1) / 2, -1, 1)))

print('═' * 50)
print('  RANSAC 3D-3D Alignment Results')
print('═' * 50)
print(f'  Total correspondences: {len(model_pts)}')
print(f'  Outliers injected:     {n_outliers} ({n_outliers/len(model_pts)*100:.0f}%)')
print(f'  Inliers found:         {inlier_mask.sum()}')
print(f'  Translation error:     {t_err:.2f} cm')
print(f'  Rotation error:        {angle_err:.2f} deg')
print('═' * 50)

# Visualize before/after alignment
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(*model_pts.T, c='blue', s=20, alpha=0.6, label='CAD model')
ax1.scatter(*scene_pts.T, c=['green' if m else 'red' for m in inlier_mask],
            s=20, alpha=0.6, label='Scene (green=inlier)')
ax1.set_title('Before Alignment\n(CAD=blue, Scene=green/red)', fontweight='bold')
ax1.legend(fontsize=7)

ax2 = fig.add_subplot(122, projection='3d')
model_aligned = (R_est @ model_pts.T).T + t_est
ax2.scatter(*model_aligned.T, c='blue', s=20, alpha=0.6, label='CAD aligned')
ax2.scatter(*scene_pts[inlier_mask].T, c='green', s=20, alpha=0.6, label='Scene inliers')
ax2.set_title('After RANSAC Alignment', fontweight='bold')
ax2.legend(fontsize=7)

plt.suptitle('FreeZe-style RANSAC 3D Registration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 🐳 Docker — Quick Reminder Before the Setup Section

The setup commands below use Docker. If you haven't seen Docker before, check the **full explainer in NB 20 (the "New to Docker?" section)**. Here's the short version:

> **Docker** runs a self-contained mini-environment on your machine. It's used here because FoundationPose requires a very specific combination of CUDA + GPU drivers that would be painful to install manually. The official NVIDIA container (`nvcr.io/nvidia/foundationpose`) ships with everything pre-configured — you just `docker pull` it and run.

**Why NVIDIA's container specifically?**

FoundationPose uses CUDA 11.8+ and custom CUDA extensions that must be compiled against the exact right GPU driver version. NVIDIA maintains an official Docker image that has all of this pre-built and tested — using it saves hours of environment debugging.

**What each command does (plain English):**

```bash
docker pull nvcr.io/nvidia/foundationpose:latest
# ↑ Downloads the pre-built FoundationPose environment image from NVIDIA's registry (~several GB, one-time)

docker run --gpus all -it nvcr.io/nvidia/foundationpose:latest
# ↑ Starts the container and gives it full access to your GPU(s)
#   --gpus all  → pass through all NVIDIA GPUs
#   -it         → open an interactive terminal inside the container
```

> **No GPU right now?** Skip the FoundationPose setup and focus on **FreeZe** — it runs without a dedicated GPU for smaller objects, and its conda setup (below) works on most machines.

---

## 7 — Practical Setup: What You Actually Need

### FoundationPose

```bash
# Official repo: https://github.com/NVlabs/FoundationPose
# Requirements: CUDA 11.8+, RTX GPU (8GB+ VRAM recommended)

# Recommended: use Docker image
docker pull nvcr.io/nvidia/foundationpose:latest

# Or conda install:
conda create -n foundationpose python=3.9
conda activate foundationpose
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
pip install trimesh open3d pyrender
```

**Inputs you need to provide**:
```
my_object/
  ├── model.obj          # 3D CAD model
  ├── model.mtl          # Material file
  └── texture.png        # Texture image
```

### FreeZe

```bash
# Official repo: https://github.com/snap-research/FreeZe
conda create -n freeze python=3.9
conda activate freeze
pip install torch open3d trimesh
# Download DINO weights (auto-downloaded on first run)
```

**Inputs you need**:
- RGB-D image (from RealSense, Azure Kinect, or ZED camera)
- CAD model (`.obj` + point cloud)
- Camera intrinsics (from your calibration!)

## 8 — Simulating a FreeZe Output

Since FreeZe requires a GPU + CUDA + real RGB-D input, we'll simulate what its output looks like and how to use it.

In [ ]:
# Simulate FreeZe output and visualize with OpenCV

def simulate_freeze_output():
    """
    Simulate what FreeZe returns: a 4x4 pose matrix.
    FreeZe output format: 4x4 homogeneous transformation matrix.
    """
    R = euler_to_R(0.25, 0.4, 0.1)
    t = np.array([0.08, -0.05, 0.45])
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

def draw_pose_axes_on_image(img, R, t, K, axis_length=0.08):
    """Draw XYZ axes at the estimated object pose."""
    rvec, _ = cv2.Rodrigues(R)
    dist = np.zeros(4)
    # Object origin
    origin_3d = np.array([[0, 0, 0]], dtype=np.float32)
    # Axis endpoints
    axes_3d = np.array([
        [axis_length, 0, 0],
        [0, axis_length, 0],
        [0, 0, axis_length]
    ], dtype=np.float32)
    origin_2d, _ = cv2.projectPoints(origin_3d, rvec, t, K, dist)
    axes_2d, _ = cv2.projectPoints(axes_3d, rvec, t, K, dist)
    o = tuple(origin_2d[0][0].astype(int))
    # Clamp to image bounds
    h, w = img.shape[:2]
    def clamp_pt(pt):
        x = int(np.clip(pt[0][0][0], 0, w-1))
        y = int(np.clip(pt[0][0][1], 0, h-1))
        return (x, y)
    px = clamp_pt(axes_2d[0:1])
    py = clamp_pt(axes_2d[1:2])
    pz = clamp_pt(axes_2d[2:3])
    out = img.copy()
    cv2.line(out, o, px, (0, 0, 255), 3)    # X = Red
    cv2.line(out, o, py, (0, 255, 0), 3)    # Y = Green
    cv2.line(out, o, pz, (255, 0, 0), 3)    # Z = Blue
    cv2.putText(out, 'X', px, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    cv2.putText(out, 'Y', py, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.putText(out, 'Z', pz, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    return out

# Camera intrinsics (realistic values)
K = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float64)

# Create synthetic scene image
scene = np.zeros((480, 640, 3), dtype=np.uint8)
scene[:] = (30, 30, 40)  # dark background
# Draw a fake object (box outline)
cv2.rectangle(scene, (260, 200), (380, 300), (100, 120, 140), -1)
cv2.rectangle(scene, (260, 200), (380, 300), (180, 200, 220), 2)
cv2.putText(scene, 'Object (RGB-D)', (265, 295), cv2.FONT_HERSHEY_SIMPLEX,
            0.5, (220, 220, 220), 1)

# Get FreeZe output
T_pose = simulate_freeze_output()
R_out = T_pose[:3, :3]
t_out = T_pose[:3, 3].astype(np.float32)

# Visualize
result = draw_pose_axes_on_image(scene, R_out, t_out, K, axis_length=0.06)

# Add info overlay
cv2.putText(result, 'FreeZe Estimated Pose', (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
cv2.putText(result, f'X={t_out[0]*100:.1f}cm  Y={t_out[1]*100:.1f}cm  Z={t_out[2]*100:.1f}cm',
            (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 255, 100), 1)
cv2.putText(result, 'RED=X  GREEN=Y  BLUE=Z', (10, 85),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
ax.set_title('Simulated FreeZe Pose Output\n(Axes show estimated 6D pose of detected object)',
             fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print('\nEstimated Pose Matrix (4x4):')
print(np.round(T_pose, 4))

## 9 — Full Method Comparison Table

| Attribute | solvePnP+ArUco | EfficientPose | FoundationPose | FreeZe |
|-----------|---------------|---------------|----------------|--------|
| **Needs training** | No | Yes (object-specific) | Yes (once, large-scale) | No |
| **Needs markers** | Yes | No | No | No |
| **Needs CAD model** | No | Sometimes | Optional | Yes |
| **Needs depth (RGB-D)** | No | No | Yes | Yes |
| **Works on unseen objects** | Only w/ markers | No | Limited | Yes |
| **GPU required** | No | Yes | Yes | Recommended |
| **Real-time capable** | Yes | Yes | ~30FPS | ~10–20FPS |
| **Accuracy (BOP benchmark)** | High (known scene) | Good | SOTA | Top-3 |
| **Best use case** | Controlled environments | Known object categories | Generalizable warehouse | Zero-shot industrial |

## 10 — When to Use Each Method

```
Q: Do you have markers or can you place them?
  YES → ArUco + PnP (fast, reliable, deterministic)
  NO ↓

Q: Do you have a fixed set of objects and training data?
  YES → EfficientPose (fastest DL inference)
  NO ↓

Q: Do you have a GPU and RGB-D camera?
  YES → FoundationPose (best overall accuracy + tracking)
  NO (or new objects daily) → FreeZe (zero-shot, no retraining ever)
```

## Recap

| Concept | Key Point |
|---------|----------|
| FoundationPose | 4 modes: model-based/free × estimation/tracking |
| Hypothesis generation | Many pose candidates ranked by transformer |
| FreeZe | Training-free, zero-shot, needs CAD + RGB-D |
| Two-branch architecture | Visual features (DINO) + Geometric features |
| Feature fusion | Fuse both types for texture + textureless robustness |
| RANSAC | Robust final alignment from noisy correspondences |
| Output format | 4×4 matrix `[R|t]` — use `cv2.Rodrigues` to get rvec |
| When to use | Use when you can't retrain and need zero-shot generalization |

In [ ]:
# ============================================================
# EXERCISE 1: Extract rvec and tvec from a 4x4 pose matrix
# ============================================================
# Given a 4x4 transformation matrix T (like FreeZe output),
# extract rvec (Rodrigues) and tvec for use with cv2.drawFrameAxes

T = np.array([
    [ 0.8660, -0.5000,  0.0000,  0.12],
    [ 0.5000,  0.8660,  0.0000, -0.05],
    [ 0.0000,  0.0000,  1.0000,  0.40],
    [ 0.0000,  0.0000,  0.0000,  1.00]
], dtype=np.float64)

# YOUR CODE HERE
# rvec = ???
# tvec = ???
# print(f'rvec shape: {rvec.shape}')
# print(f'tvec: {tvec}')

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# R = T[:3, :3]
# tvec = T[:3, 3]
# rvec, _ = cv2.Rodrigues(R)
# print(f'rvec shape: {rvec.shape}')   # (3, 1)
# print(f'tvec: {tvec}')               # [0.12, -0.05, 0.40]

In [ ]:
# ============================================================
# EXERCISE 2: Implement a simple Kabsch alignment (no RANSAC)
# ============================================================
# Use the compute_rigid_transform() function defined above
# to find [R, t] for 3D-3D correspondences WITHOUT outliers.
# Then verify the recovered pose by transforming src → dst.

np.random.seed(99)
src = np.random.rand(30, 3) * 0.3
R_known = euler_to_R(0.7, 0.2, -0.4)
t_known = np.array([0.05, 0.10, 0.20])
dst = (R_known @ src.T).T + t_known + np.random.randn(30, 3) * 0.001  # tiny noise

# YOUR CODE HERE
# R_recovered, t_recovered = ???
# t_error_cm = ???
# print(f'Translation error: {t_error_cm:.3f} cm')

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# R_recovered, t_recovered = compute_rigid_transform(src, dst)
# t_error_cm = np.linalg.norm(t_recovered - t_known) * 100
# rot_diff = R_known @ R_recovered.T
# angle_err = np.degrees(np.arccos(np.clip((np.trace(rot_diff)-1)/2, -1, 1)))
# print(f'Translation error: {t_error_cm:.3f} cm')
# print(f'Rotation error:    {angle_err:.4f} deg')